<a href="https://colab.research.google.com/github/AhsanullahCS/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhsanullahCS/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

I will prioritize pages that are both stale and visibly active.

A page is considered stale when it has not been updated for at least 180 days.
A page is considered visible when it has at least 500 impressions in the 90-day window.

The score gives priority to pages that satisfy both conditions and have more impressions.
This is a transparent decision-support baseline, not a prediction of future performance.

### Reason code

- `stale_visible_page` — the page is at least 180 days since its last update and has at least 500 impressions in the 90-day window.

### Action

- `refresh` — review the page for a possible content refresh.
- `monitor` — does not meet both baseline conditions.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the dataset uploaded to Colab
DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nRequired columns:")
print([
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
])

print("\nMissing values in rule columns:")
print(
    df[
        [
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ].isna().sum()
)

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Required columns:
['content_id', 'client_id', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']

Missing values in rule columns:
days_since_last_update    0
impressions_90d         

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# Load dataset
# =========================================================

DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values in rule columns:")

rule_columns = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

print(df[rule_columns].isna().sum())

Dataset loaded successfully.
Shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Missing values in rule columns:
content_id                0
client_id                 0
days_since_last_update    0
impressions_90d           0
avg_position              0
ctr                       0
dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# =========================================================
# SIGNAL 1 — STALENESS
# =========================================================

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=[
        "<=90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          mean_impressions_90d=("impressions_90d", "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — STALENESS")
print("=" * 70)

display(staleness_table)

SIGNAL 1 — STALENESS


,staleness_bucket,n,median_impressions_90d,mean_impressions_90d
0,<=90 days,20655,472.0,4219.161317
1,91-180 days,9171,1692.0,7486.665140
2,181-365 days,169,16.0,1206.893491
3,365+ days,5,2.0,8.200000


### Signal 1 verdict — STALENESS

I measured impressions across four staleness buckets and included the number of observations (`n`) in every bucket.

**Verdict: [WRITE ONE: CONFIRMED / OPPOSITE / MIXED / FALSE]**

The verdict is based only on the observed bucket table. The purpose of this check is to determine whether staleness provides a useful directional signal for a refresh-review queue.

In [5]:
# =========================================================
# SIGNAL 2 — VISIBILITY / VOLUME
# =========================================================

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 499, 2999, np.inf],
    labels=[
        "<100",
        "100-499",
        "500-2999",
        "3000+"
    ]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_staleness=("days_since_last_update", "median"),
          mean_staleness=("days_since_last_update", "mean")
      )
      .reset_index()
)

print("SIGNAL 2 — VISIBILITY / VOLUME")
print("=" * 70)

display(visibility_table)

SIGNAL 2 — VISIBILITY / VOLUME


,visibility_bucket,n,median_staleness,mean_staleness
0,<100,7994,20.0,33.120090
1,100-499,5280,22.0,45.062689
2,500-2999,8443,22.0,49.653085
3,3000+,8283,25.0,55.660389


In [6]:
# =========================================================
# Prepare rule inputs
# =========================================================

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# Missing values cannot establish either condition.
# Treat them as not meeting the condition.
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)
df["impressions_90d"] = df["impressions_90d"].fillna(0)

# Stale = at least 180 days since update
df["stale"] = (
    df["days_since_last_update"] >= 180
).astype(int)

# Visible = at least 500 impressions
df["visible"] = (
    df["impressions_90d"] >= 500
).astype(int)

print("Stale pages:", df["stale"].sum())
print("Visible pages:", df["visible"].sum())
print(
    "Stale + visible pages:",
    ((df["stale"] == 1) & (df["visible"] == 1)).sum()
)

Stale pages: 174
Visible pages: 16726
Stale + visible pages: 17


## 2. Build the ranked queue

### Score

The score is:

**stale × visible × impressions_90d**

This means a page receives a positive score only when both baseline conditions are satisfied.

Among pages that are both stale and visible, pages with more observed 90-day impressions receive a higher score.

### Ranking

Pages are sorted from the highest score to the lowest score.

### Reason code

Only one primary reason code is assigned to each row:

- `stale_visible_page`
- `general_review`

### Action

- `refresh`
- `monitor`

In [7]:
# =========================================================
# BUILD BASELINE SCORE
# =========================================================

# Score
df["baseline_score"] = (
    df["stale"]
    * df["visible"]
    * df["impressions_90d"]
)

# One reason code
df["reason_code"] = np.where(
    (df["stale"] == 1) & (df["visible"] == 1),
    "stale_visible_page",
    "general_review"
)

# Action
df["action"] = np.where(
    df["baseline_score"] > 0,
    "refresh",
    "monitor"
)

# Rank everything
df = df.sort_values(
    by=["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["baseline_rank"] = np.arange(1, len(df) + 1)

print("Ranking completed.")

print("\nAction counts:")
print(df["action"].value_counts())

print("\nTop 10:")
display(
    df[
        [
            "baseline_rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ].head(10)
)

Ranking completed.

Action counts:
action
monitor    29983
refresh       17
Name: count, dtype: int64

Top 10:


,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,61678,stale_visible_page,refresh,194,61678
1,2,content_7368877ea310,59472,stale_visible_page,refresh,194,59472
2,3,content_1bfaa38ff26c,25715,stale_visible_page,refresh,194,25715
3,4,content_0a91db491d14,13299,stale_visible_page,refresh,193,13299
4,5,content_5feee3994adb,7812,stale_visible_page,refresh,194,7812
5,6,content_c2d929d83eaa,7558,stale_visible_page,refresh,193,7558
6,7,content_b16bd7307b39,4590,stale_visible_page,refresh,194,4590
7,8,content_fe16a55cd13d,4556,stale_visible_page,refresh,194,4556
8,9,content_ecb6215e79fd,4429,stale_visible_page,refresh,194,4429
9,10,content_928af3e22c80,1697,stale_visible_page,refresh,193,1697


In [8]:
# =========================================================
# WRITE RANKED QUEUE TO CSV
# =========================================================

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue = df[output_columns].copy()

# Colab output directory
OUTPUT_DIR = Path("/content/work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV successfully written.")
print("Path:", OUTPUT_PATH)
print("Rows:", len(queue))
print("Columns:", len(queue.columns))

CSV successfully written.
Path: /content/work/outputs/baseline_action_score.csv
Rows: 30000
Columns: 10


## 3. Top-20 review

The top 20 rows are the highest-ranked pages produced by the baseline.

For each row I record:

1. The recommended action.
2. The reason code.
3. A confidence note.
4. What would make the recommendation wrong.

These are decision-support notes rather than claims that a refresh will definitely improve performance.

In [9]:
# =========================================================
# TOP-20 REVIEW
# =========================================================

top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["action"] == "refresh",
    "Moderate confidence: both rule conditions are directly observed.",
    "Low confidence: the row does not meet both baseline conditions."
)

top20["what_would_make_it_wrong"] = (
    "The page may still be accurate and useful; "
    "high impressions do not by themselves prove that a refresh is needed."
)

top20_review = top20[
    [
        "baseline_rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,baseline_rank,content_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,refresh,stale_visible_page,61678,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
1,2,content_7368877ea310,refresh,stale_visible_page,59472,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
2,3,content_1bfaa38ff26c,refresh,stale_visible_page,25715,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
3,4,content_0a91db491d14,refresh,stale_visible_page,13299,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
4,5,content_5feee3994adb,refresh,stale_visible_page,7812,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
5,6,content_c2d929d83eaa,refresh,stale_visible_page,7558,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
6,7,content_b16bd7307b39,refresh,stale_visible_page,4590,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
7,8,content_fe16a55cd13d,refresh,stale_visible_page,4556,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
8,9,content_ecb6215e79fd,refresh,stale_visible_page,4429,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...
9,10,content_928af3e22c80,refresh,stale_visible_page,1697,Moderate confidence: both rule conditions are ...,The page may still be accurate and useful; hig...


In [10]:
# =========================================================
# TOP-20 EVIDENCE
# =========================================================

top20_evidence = queue.head(20)[
    [
        "baseline_rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
].copy()

display(top20_evidence)

,baseline_rank,content_id,action,reason_code,baseline_score,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,refresh,stale_visible_page,61678,194,61678,19.7,0.15
1,2,content_7368877ea310,refresh,stale_visible_page,59472,194,59472,24.8,0.13
2,3,content_1bfaa38ff26c,refresh,stale_visible_page,25715,194,25715,22.2,0.23
3,4,content_0a91db491d14,refresh,stale_visible_page,13299,193,13299,10.5,0.49
4,5,content_5feee3994adb,refresh,stale_visible_page,7812,194,7812,39.0,0.01
5,6,content_c2d929d83eaa,refresh,stale_visible_page,7558,193,7558,17.9,0.20
6,7,content_b16bd7307b39,refresh,stale_visible_page,4590,194,4590,31.0,0.00
7,8,content_fe16a55cd13d,refresh,stale_visible_page,4556,194,4556,16.4,0.33
8,9,content_ecb6215e79fd,refresh,stale_visible_page,4429,194,4429,25.3,0.38
9,10,content_928af3e22c80,refresh,stale_visible_page,1697,193,1697,15.8,0.12


## 4. Weak picks + leakage check

### Weak picks

The baseline is intentionally simple, so some top-ranked pages can still be wrong.

A page can be old and visible while its existing content remains accurate and useful.

Therefore the `refresh` action means "send for human review", not "automatically rewrite the page."

### Leakage check

The baseline uses only current observed inputs:

- `days_since_last_update`
- `impressions_90d`

I do not use future-window outcomes or label-derived fields to calculate the score.

In [11]:
# =========================================================
# WEAK PICKS
# =========================================================

# Look for flagged pages that have relatively low visibility.
weak_picks = queue[
    (queue["action"] == "refresh") &
    (queue["impressions_90d"] < 1000)
].head(10)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "action",
            "reason_code",
            "baseline_score",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
)

Potential weak picks:


,baseline_rank,content_id,action,reason_code,baseline_score,days_since_last_update,impressions_90d
12,13,content_7f116ae1f6f5,refresh,stale_visible_page,954,301,954
13,14,content_77d4d5930e5e,refresh,stale_visible_page,828,194,828
14,15,content_72496874f806,refresh,stale_visible_page,821,301,821
15,16,content_6226ee6adc91,refresh,stale_visible_page,545,183,545
16,17,content_074ba6ead17b,refresh,stale_visible_page,533,183,533


In [12]:
# =========================================================
# LEAKAGE CHECK
# =========================================================

# Columns actually used by our baseline score
used_rule_columns = {
    "days_since_last_update",
    "impressions_90d"
}

# Known label/future-related columns that must NOT be used
forbidden_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

# Check that forbidden fields were not used
leaked_columns = used_rule_columns.intersection(
    forbidden_columns
)

print("Rule input columns:")
print(sorted(used_rule_columns))

print("\nForbidden label/future columns:")
print(sorted(forbidden_columns))

print("\nColumns accidentally used from forbidden set:")
print(sorted(leaked_columns))

assert len(leaked_columns) == 0, (
    "Leakage detected!"
)

print("\nPASS — No forbidden label-derived inputs are used.")

# Explicitly verify that no future-window field is part of the score
score_formula_columns = {
    "stale",
    "visible",
    "impressions_90d"
}

print("\nScore uses:")
print(sorted(score_formula_columns))

print("\nPASS — Score uses current observed fields only.")

Rule input columns:
['days_since_last_update', 'impressions_90d']

Forbidden label/future columns:
['is_declining_label', 'trend_direction', 'trend_pct']

Columns accidentally used from forbidden set:
[]

PASS — No forbidden label-derived inputs are used.

Score uses:
['impressions_90d', 'stale', 'visible']

PASS — Score uses current observed fields only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.